# TCC - Analise Preditiva de Risco Academico Escolar
## CRISP-DM: 4. Modeling

## O que esta fase responde

A fase de Modeling responde:

- quais modelos serao testados;
- quais colunas entram como entrada do modelo;
- qual e o alvo de regressao;
- qual e o alvo de classificacao;
- como treino, validacao e teste sao separados;
- qual criterio escolhe o melhor modelo;
- como o modelo final e treinado depois da escolha.

## Tarefas de aprendizagem do projeto

| Tarefa | Alvo | Tipo | Metrica de escolha |
|---|---|---|---|
| Previsao de nota | `target_nota_proxima` | Regressao | menor MAE |
| Alerta de risco | `target_risco_proxima` | Classificacao binaria | maior F1 |

A regressao estima a proxima nota. A classificacao estima se a proxima nota ficara abaixo de 6.0. As duas tarefas sao treinadas na mesma funcao porque compartilham o mesmo dataset temporal, mas possuem alvos, candidatos e metricas diferentes.

Isso significa que o projeto nao faz primeiro a regressao para depois derivar a classificacao. As duas trilhas sao avaliadas em paralelo sobre o mesmo recorte temporal da base.

## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar funcoes reais da pipeline

A demonstracao usa as mesmas funcoes do projeto para evitar divergencia entre notebook e codigo de producao academica.

In [ ]:
from school_predictor.pipeline.config import PipelinePaths
from school_predictor.pipeline.data import load_source_tables
from school_predictor.pipeline.dataset import build_prediction_dataset, select_model_columns
from school_predictor.pipeline.modeling import (
    build_classification_baselines,
    build_classification_candidates,
    build_regression_baselines,
    build_regression_candidates,
    temporal_split,
    train_and_evaluate,
)
from school_predictor.pipeline.orchestration import resolve_mode_settings, run_real_pipeline

## Definir modo de demonstracao

O projeto possui dois modos operacionais:

- `previsao_nota`: usa historico minimo padrao 1.
- `alerta_risco`: usa historico minimo padrao 2.

Mesmo quando um modo e chamado separadamente, a funcao de treinamento calcula regressao e classificacao para manter os dois sinais comparaveis no relatorio tecnico.

Na pratica, o `mode` altera principalmente o `min_history` padrao, o dataset salvo naquele diretório e os artefatos operacionais. A etapa central de modelagem continua produzindo os dois sinais: nota prevista e risco previsto.

### Exemplo fixo da diferenca entre os modos

A tabela abaixo resume como cada modo muda o volume de dados e quem venceu na validação:

| modo | min_history | linhas_modelagem | anos_disponiveis | vencedor_regressao | vencedor_classificacao |
| --- | --- | --- | --- | --- | --- |
| previsao_nota | 1 | 106721 | 2020-2025 | RandomForest | Baseline media_duas_ultimas |
| alerta_risco | 2 | 50065 | 2021-2025 | HistGradientBoosting | HistGradientBoosting |

In [ ]:
MODE = "previsao_nota"  # alternativas: "previsao_nota" ou "alerta_risco"
MODE, MIN_HISTORY = resolve_mode_settings(MODE, min_history=None)

MODE, MIN_HISTORY

### O que este modo altera na pratica

A saida desta celula mostra qual contexto operacional esta sendo simulado. Quando o modo muda, o principal efeito visivel e o corte de `min_history`, que altera quantas linhas entram no dataset daquele artefato.

O ponto importante e que o treinamento central continua avaliando os dois sinais em paralelo: nota prevista e risco previsto.

## Carregar ou construir dataset de modelagem

Se o dataset da pipeline ja existir em `artifacts/pipeline/<modo>/student_prediction_dataset.csv`, ele pode ser carregado diretamente. Caso contrario, ele e reconstruido a partir dos CSVs canonicos.

Como o `mode` afeta o `min_history`, os datasets de `previsao_nota` e `alerta_risco` podem ter volumes diferentes mesmo vindo das mesmas bases brutas.

In [ ]:
paths = PipelinePaths.from_project_root(PROJECT_ROOT, min_history=MIN_HISTORY, mode=MODE)

if paths.dataset_csv.exists():
    dataset = pd.read_csv(paths.dataset_csv, encoding="utf-8", low_memory=False)
else:
    source_tables = load_source_tables(paths)
    dataset = build_prediction_dataset(source_tables, min_history=MIN_HISTORY)

pd.DataFrame([{
    "modo": MODE,
    "min_history": MIN_HISTORY,
    "linhas": len(dataset),
    "colunas": len(dataset.columns),
    "alunos": dataset["IDAluno"].nunique() if "IDAluno" in dataset.columns else None,
    "disciplinas": dataset["NomeDisciplina"].nunique() if "NomeDisciplina" in dataset.columns else None,
}])

### O que o dataset de modelagem representa aqui

Ao executar a celula acima, voce passa a enxergar o tamanho real da base que chegou ao treinamento para este modo. Isso permite comparar como a exigencia de historico minimo altera a quantidade de alunos, disciplinas e observacoes disponiveis.

Didaticamente, esta e a ponte entre Data Preparation e Modeling: a pergunta agora deixa de ser como o dado foi montado e passa a ser se ele e suficiente para treinar candidatos competitivos.

## Separacao temporal: treino, validacao e teste

A separacao nao e aleatoria. O projeto usa anos letivos:

- anos mais antigos para treino;
- penultimo ano para validacao;
- ultimo ano para teste.

Isso evita que informacoes do futuro vazem para o treinamento e deixa a avaliacao mais parecida com o uso real na escola.

Quando existem apenas dois anos letivos disponiveis, o codigo usa um fallback: o ultimo ano passa a servir como validacao e teste, porque nao ha um terceiro ano para separar melhor os papeis. Quando ha tres ou mais anos, o comportamento ideal e treino em anos mais antigos, validacao no penultimo e teste no ultimo.

### Exemplo fixo do split temporal

Aqui fica visível que a divisão respeita o tempo, separando passado para treino e futuro para validação/teste:

| modo | anos_treino | linhas_treino | ano_validacao | linhas_validacao | ano_teste | linhas_teste |
| --- | --- | --- | --- | --- | --- | --- |
| previsao_nota | 2020, 2021, 2022, 2023 | 52759 | 2024 | 27435 | 2025 | 26527 |
| alerta_risco | 2021, 2022, 2023 | 23134 | 2024 | 13688 | 2025 | 13243 |

In [ ]:
split = temporal_split(dataset)

pd.DataFrame([
    {"conjunto": "treino", "anos": split["train_years"], "linhas": len(split["train"])},
    {"conjunto": "validacao", "anos": split["validation_years"], "linhas": len(split["validation"])},
    {"conjunto": "teste", "anos": split["test_years"], "linhas": len(split["test"])},
])

### O que aconteceu com os dados apos o split temporal

A base foi quebrada em passado, validacao e teste sem sorteio aleatorio. Isso faz com que cada conjunto tenha um papel claro:
- treino para ajustar os candidatos;
- validacao para escolher o melhor;
- teste para medir o desempenho final em um ano ainda nao visto.

Quando esta etapa termina, o dado ja nao e mais um bloco unico. Ele vira um experimento temporal controlado.

## Selecionar entradas dos modelos

Identificadores diretos e alvos nao entram como features. O modelo recebe historico, contexto academico, faltas, pagamentos agregados, professor e sinais derivados.

Na execucao real, a funcao `train_and_evaluate` ainda remove colunas sem variabilidade ou totalmente vazias no conjunto de treino antes de ajustar os candidatos. Isso evita manter features sem sinal util.

### Exemplo fixo da entrada dos modelos

Esta é uma pequena amostra da linha que o modelo recebe para aprender e prever a próxima nota/risco:

| NomePeriodo | NomeSerie | NomeDisciplina | NomeEtapa | ValorMedia | nota_anterior_1 | faltas_acumuladas | pagamentos_pendentes_ano | professor_disponivel | target_nota_proxima | target_risco_proxima |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 1ª Série | Arte | 2º BIMESTRE | 8.8 | 10.0 | 1.0 | 0.0 | 1 | 8.1 | 0 |
| 2025 | 1ª Série | Arte | 3º BIMESTRE | 8.1 | 8.8 | 3.0 | 0.0 | 1 | 10.0 | 0 |
| 2025 | 1ª Série | Biologia | 2º BIMESTRE | 7.0 | 6.8 | 7.0 | 0.0 | 1 | 7.4 | 0 |
| 2025 | 1ª Série | Biologia | 3º BIMESTRE | 7.4 | 7.0 | 14.0 | 0.0 | 1 | 7.0 | 0 |
| 2025 | 1ª Série | Biologia 1 | 2º BIMESTRE | 7.0 | 8.5 | 7.0 | 0.0 | 1 | 5.6 | 1 |

In [ ]:
feature_columns, categorical_columns = select_model_columns(dataset)

features_frame = pd.DataFrame({
    "feature": feature_columns,
    "tratamento": ["categorica" if column in categorical_columns else "numerica_ou_ordinal" for column in feature_columns],
})

features_frame

### O que esta tabela mostra sobre a entrada do modelo

Depois de executar a celula acima, fica visivel exatamente quais colunas seguem para o algoritmo. O dado cru foi reduzido a um conjunto de sinais historicos, contextuais e categoricos que podem ser codificados e aprendidos.

Isso ajuda muito na defesa, porque transforma a pergunta "o modelo olha para o que?" em uma resposta objetiva e auditavel.

## Modelos candidatos

O projeto compara dois tipos de modelos de arvore:

- Random Forest: robusto, bom baseline supervisionado forte, lida bem com relacoes nao lineares e mistura de variaveis.
- HistGradientBoosting: modelo de boosting eficiente, tende a capturar padroes tabulares mais finos quando ha dados suficientes.

Tambem entram baselines simples baseados nas notas anteriores. Isso evita afirmar que o modelo e util se ele nao superar regras simples.

Na implementacao atual, Random Forest usa codificacao categórica por one-hot encoding, enquanto HistGradientBoosting usa codificacao ordinal. A escolha final nao e fixa: qualquer candidato, inclusive baseline, pode vencer na validacao.

### Exemplo fixo dos candidatos comparados

Abaixo está um resumo simples do que é comparado nesta fase:

| familia | candidato | codificacao_categorica | criterio_de_escolha |
| --- | --- | --- | --- |
| Regressao | RandomForestRegressor | OneHotEncoder | menor MAE |
| Regressao | HistGradientBoostingRegressor | OrdinalEncoder | menor MAE |
| Classificacao | RandomForestClassifier | OneHotEncoder | maior F1 |
| Classificacao | HistGradientBoostingClassifier | OrdinalEncoder | maior F1 |
| Baseline | ultima_nota / medias_historicas | nao se aplica | comparacao honesta |

In [ ]:
regression_candidates = build_regression_candidates(feature_columns, categorical_columns)
classification_candidates = build_classification_candidates(feature_columns, categorical_columns)

pd.DataFrame([
    {"familia": "regressao", "candidato": name}
    for name in regression_candidates.keys()
] + [
    {"familia": "classificacao", "candidato": name}
    for name in classification_candidates.keys()
] + [
    {"familia": "baseline_regressao", "candidato": name}
    for name in build_regression_baselines(dataset).keys()
] + [
    {"familia": "baseline_classificacao", "candidato": name}
    for name in build_classification_baselines(dataset).keys()
])

### O que os candidatos significam quando comparados lado a lado

A saida acima mostra que a modelagem nao parte do pressuposto de que um algoritmo complexo sempre sera melhor. Baselines simples entram na disputa com os modelos supervisionados.

Na pratica, isso quer dizer que o projeto so considera ganho real quando um candidato supera regras que ja seriam razoaveis sem aprendizado de maquina.

## Por que nao usar apenas ifs?

Regras como `se ultima nota < 6, entao risco` sao baselines importantes, mas sao limitadas. Elas olham para um corte fixo e ignoram combinacoes como queda de tendencia, historico anterior, faltas, media da coorte, pagamentos agregados, serie, etapa e disciplina.

A classificacao aprende padroes combinados. Um aluno com nota atual acima de 6 pode ainda receber alerta se o historico indicar queda, acumulado de faltas ou comportamento parecido com alunos que cairam abaixo da linha na etapa seguinte. Da mesma forma, um aluno com nota baixa pode nao ser classificado como risco alto se a tendencia e o contexto indicarem recuperacao.

## Criterios de escolha

- Regressao: o melhor candidato e o que tiver menor MAE na validacao.
- Classificacao: o melhor candidato e o que tiver maior F1 na validacao.

O MAE mede, em media, quantos pontos a previsao de nota erra. O F1 equilibra precision e recall, evitando olhar apenas para acerto geral quando o evento de risco pode ser menor que a quantidade de alunos sem risco.

### Exemplo fixo dos criterios e vencedores

A escolha final depende da métrica principal de cada tarefa e pode até eleger um baseline:

| modo | alvo | fonte_vencedora | vencedor | metrica_usada | valor_na_validacao |
| --- | --- | --- | --- | --- | --- |
| previsao_nota | regressao | modelo | RandomForest | MAE | 0.8152 |
| previsao_nota | classificacao | baseline | media_duas_ultimas | F1 | 0.7359 |
| alerta_risco | regressao | modelo | HistGradientBoosting | MAE | 0.8124 |
| alerta_risco | classificacao | modelo | HistGradientBoosting | F1 | 0.7564 |

## Executar treinamento sob demanda

Por padrao, o treinamento fica desligado para manter o notebook leve e seguro para GitHub. Para rodar localmente, altere `RUN_TRAINING = True`.

A funcao `train_and_evaluate` executa a selecao dos candidatos, compara modelos e baselines na validacao, re-treina os vencedores com treino + validacao e retorna metricas, predicoes e analise de erro. Ela nao salva arquivos. Para salvar artefatos iguais aos da CLI, use `run_real_pipeline` no bloco opcional seguinte.

### Exemplo fixo do que sai quando o treinamento termina

Quando a fase de modelagem termina, estes são os principais artefatos deixados para avaliação e deployment:

| artefato_gerado | o_que_contem |
| --- | --- |
| student_prediction_dataset.csv | base final de modelagem, já com features e alvos |
| student_prediction_predictions.csv | predições finais sobre o conjunto de teste |
| student_prediction_report.txt | ranking de candidatos, métricas e vencedores |
| student_prediction_model.pkl | pipeline treinada escolhida para o modo |
| error_analysis_by_subject.csv | erro agregado por disciplina |
| error_analysis_by_series.csv | erro agregado por série |
| error_analysis_by_score_band.csv | erro agregado por faixa de nota |

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    training_output = train_and_evaluate(dataset, feature_columns, categorical_columns)
    metrics = training_output["metrics"]

    display(pd.DataFrame([
        {"grupo": "regressao_modelo", **metrics["regression_model"]},
        {"grupo": "regressao_baseline", **metrics["regression_baseline"]},
    ]))
    display(pd.DataFrame([
        {"grupo": "classificacao_modelo", **metrics["classification_model"]},
        {"grupo": "classificacao_baseline", **metrics["classification_baseline"]},
    ]))
    display(pd.DataFrame([
        {"tipo": "regressao", **metrics["selected_regression_candidate"]},
        {"tipo": "classificacao", **metrics["selected_classification_candidate"]},
    ]))
else:
    print("Treinamento nao executado. Altere RUN_TRAINING para True para rodar localmente.")

### O que acontece quando o treinamento roda

Se voce ativar `RUN_TRAINING`, a pipeline vai testar todos os candidatos na validacao, escolher o melhor por metrica da tarefa e depois re-treinar esse vencedor com treino + validacao.

A saida que aparece depois disso nao e apenas um numero final. Ela mostra:
- desempenho do modelo vencedor;
- desempenho do baseline de referencia;
- quem venceu em regressao;
- quem venceu em classificacao.

## Opcional: rodar a pipeline do modo e salvar artefatos

Este bloco reproduz a execucao operacional da CLI para um modo especifico. Ele salva dataset, modelo, predicoes e relatorio tecnico em `artifacts/pipeline/<modo>/`. Use apenas localmente.

In [ ]:
RUN_AND_SAVE_ARTIFACTS = False

if RUN_AND_SAVE_ARTIFACTS:
    output = run_real_pipeline(project_root=PROJECT_ROOT, mode=MODE, min_history=MIN_HISTORY)
    print(output["paths"].output_dir)

## Resultado da fase

Ao final da modelagem, o projeto possui:

- candidato escolhido para regressao, por menor MAE na validacao;
- candidato escolhido para classificacao, por maior F1 na validacao;
- comparacao com baselines simples;
- re-treinamento usando treino + validacao;
- teste final no ano mais recente;
- predicoes com nota prevista, risco previsto, erro absoluto e acerto do risco.

Esses resultados seguem para a fase de Evaluation, onde as metricas sao interpretadas e os erros sao analisados por disciplina, serie e faixa de nota.

O uso da palavra candidato e importante aqui porque o vencedor pode ser um modelo de arvore ou um baseline simples, dependendo do desempenho observado na validacao.

## Leitura didatica da fase

No fim deste notebook, o dado ja passou por uma mudanca conceitual importante:
- antes, ele era apenas uma base temporal preparada;
- agora, ele virou experimento comparativo entre candidatos.

A fase de Modeling, portanto, nao entrega apenas um modelo salvo. Ela entrega uma decisao justificavel sobre qual candidato merece seguir para o teste final.